이 노트북을 실행하는 데 필요한 라이브러리(표준 라이브러리 제외)
- torch
- numpy
- matplotlib
- groq
- sentence_transformers
- transformers

- 실습 기본 환경 설정


In [1]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

CUDA를 사용합니다.


# 12-4 검색 증강 생성으로 환각 줄이기

본 노트북은 본문 12-4절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 문장 임베딩 모델로 근거 문서를 벡터화하고 질문과 가장 유사한 문서 검색
- Groq 클라이언트로 LLM 호출
- 검색 결과를 프롬프트에 끼워 넣는 RAG 프롬프트 구성과 통합 파이프라인
- RAG 적용 여부에 따른 답변 비교([표 12-11])

본문의 모델 16의 구현에 해당되며, 모델 16의 제시문은 다음과 같다.

> **모델 16. 진실한 인공지능 대화 서비스**
>
> LLM에 외부 지식을 검색해 함께 입력하는 방식으로, 환각을 줄이고 최신 정보에 근거해 답변하는 서비스를 만들어 본다.

> 본문 [표 12-10]은 RAG 파이프라인의 단계별 대표 도구를 정리한다.

## 준비

- Groq API 키를 환경 변수로 설정한다. 키가 없으면 뒤에서 로컬 LLM으로 대체한다.

In [2]:
# 참고 - Groq API 키 설정
# 발급받은 키가 있으면 아래 변수에 직접 넣거나, 셸에서 환경 변수로 설정한다.
#   리눅스/macOS:  export GROQ_API_KEY='발급받은 키'
#   주피터 노트북: os.environ['GROQ_API_KEY'] = '발급받은 키'
# 키가 비어 있으면 이 노트북은 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
import os

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')  # 'groq_key' 자리에 키 입력
USE_LOCAL_LLM = not bool(GROQ_API_KEY)
print(f'로컬 LLM 폴백 모드: {USE_LOCAL_LLM}')

로컬 LLM 폴백 모드: True


- 답변의 근거가 될 짧은 글 여섯 편을 딕셔너리로 둔다.
    - 실제 서비스라면 문서 저장소나 벡터 데이터베이스에서 가져오는 부분이다.

In [3]:
# 참고 - LLM·트랜스포머 관련 짧은 글 6편
documents = [
    {'title': '트랜스포머 아키텍처',
     'content': (
         '트랜스포머는 2017년 구글이 발표한 신경망 구조로, 어텐션 메커니즘만으로 '
         '순차 데이터를 처리한다. 인코더와 디코더가 모두 셀프 어텐션과 피드포워드 '
         '계층의 반복으로 구성되며, 위치 정보는 위치 인코딩을 통해 더한다. 이후 '
         '거의 모든 대규모 언어 모델의 기본 구조가 되었다.')},
    {'title': 'GPT 시리즈',
     'content': (
         'GPT는 OpenAI가 공개한 디코더 전용 트랜스포머 계열의 언어 모델 시리즈로, '
         'GPT-3(2020)에서 1,750억 파라미터로 규모를 크게 키워 화제가 됐다. GPT-4는 '
         '멀티모달 입력을 지원하며 ChatGPT 서비스의 기반 모델로 사용된다. 대규모 '
         '사전 학습 후 지시어 미세 조정과 RLHF로 정렬되는 파이프라인을 따른다.')},
    {'title': 'LLaMA',
     'content': (
         'LLaMA는 Meta가 2023년부터 공개한 오픈 가중치 대규모 언어 모델 시리즈다. '
         'LLaMA 2와 LLaMA 3로 이어지며 연구·상용 모두 사용 가능한 라이선스로 배포돼 '
         '오픈소스 LLM 생태계의 표준이 됐다. 한국어 특화 파생 모델인 Bllossom도 '
         'LLaMA 3 계열을 기반으로 한다.')},
    {'title': 'RAG(검색 증강 생성)',
     'content': (
         'RAG는 2020년 메타(Meta) AI 연구팀이 발표한 기법으로, LLM의 환각과 학습 후 '
         '정보 부재 문제를 외부 문서 검색으로 보완한다. 파이프라인은 (1) 문서를 임베딩 '
         '벡터로 변환해 저장, (2) 질문을 임베딩해 유사한 문서를 검색, (3) 검색된 문서를 '
         '컨텍스트로 LLM에 전달해 답변을 생성하는 세 단계로 구성된다.')},
    {'title': 'Groq',
     'content': (
         'Groq는 LPU(Language Processing Unit)라는 AI 추론 전용 칩을 개발한 '
         '스타트업이다. 오픈소스 LLM의 API 서비스를 함께 제공하며, 호출 인터페이스가 '
         'OpenAI API와 동일해 코드 호환성이 높다. 무료 티어에서도 학습·실험용으로 '
         '충분한 사용량을 제공해 RAG 같은 빠른 응답이 필요한 시스템에 적합하다.')},
    {'title': '파인튜닝과 프롬프트 엔지니어링',
     'content': (
         '파인튜닝은 사전 학습 모델을 작업 데이터로 추가 학습해 모델의 동작 자체를 '
         '바꾸는 방법이다. 반면 프롬프트 엔지니어링은 모델은 그대로 두고 입력 프롬프트를 '
         '잘 구성해 원하는 출력을 끌어내는 방법이다. RAG는 후자의 발전된 형태로, 모델은 '
         '그대로 두고 외부 문서를 동적으로 컨텍스트에 끼워 넣어 답변을 보강한다.')},
]
print(f'문서 개수: {len(documents)}')
for doc in documents:
    print(f"- {doc['title']} ({len(doc['content'])}자)")

문서 개수: 6
- 트랜스포머 아키텍처 (152자)
- GPT 시리즈 (183자)
- LLaMA (163자)
- RAG(검색 증강 생성) (182자)
- Groq (187자)
- 파인튜닝과 프롬프트 엔지니어링 (174자)


## 문서 임베딩과 검색

- 한국어 문장 유사도에 특화된 모델을 `SentenceTransformer`로 불러와 문서를 벡터화한다.
    - 임베딩을 정규화해 두면 코사인 유사도를 행렬곱 한 번으로 계산할 수 있다.

In [4]:
######################################################################################
# 코드 12-14 - 문서 임베딩 생성
######################################################################################

import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')


def build_index(documents, model):
    texts = [doc['content'] for doc in documents]
    embeddings = model.encode(texts, convert_to_tensor=True)
    # L2 정규화: 코사인 유사도를 내적으로 계산하기 위한 전처리
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings


doc_embeddings = build_index(documents, embed_model)
print(f'임베딩 shape: {doc_embeddings.shape}')   # (문서 수, 임베딩 차원)

/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20004.47it/s]

임베딩 shape: torch.Size([6, 768])


- 질문도 같은 방식으로 임베딩하고 정규화한 뒤, 문서 임베딩과의 행렬곱으로 모든 문서의 유사도를 한 번에 구한다.

In [5]:
######################################################################################
# 코드 12-15 - 질문과 가장 유사한 문서 검색
######################################################################################

import torch

def retrieve(query, documents, doc_embeddings, embed_model, top_k=2):
    # doc_embeddings는 build_index()에서 이미 L2 정규화된 상태로 전달받는다고 가정.
    # 질문도 같은 방식으로 임베딩하고 정규화한다.
    query_embedding = embed_model.encode(query, convert_to_tensor=True)
    # 1차원 벡터이므로 dim=0 (배치 텐서면 dim=1 또는 차원과 무관하게 dim=-1)
    query_embedding = F.normalize(query_embedding, p=2, dim=0)
    # 행렬 곱으로 코사인 유사도 계산 (정규화된 벡터의 내적 = 코사인 유사도)
    scores = torch.matmul(doc_embeddings, query_embedding)
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    return ([documents[i] for i in top_indices],
            [scores[i].item() for i in top_indices])


retrieved_docs, scores = retrieve(
    '트랜스포머 모델은 어떤 구조인가요?', documents, doc_embeddings, embed_model,
)
for doc, score in zip(retrieved_docs, scores):
    print(f'[유사도: {score:.4f}] {doc["title"]}')

[유사도: 0.3957] 트랜스포머 아키텍처
[유사도: 0.3523] GPT 시리즈


## 생성

- `GROQ_API_KEY`가 있으면 실제 `groq.Groq` 클라이언트를 사용하고, 없으면 로컬 LLM으로 대체한다.

In [6]:
# 참고 - 생성 클라이언트 (실제 Groq API 또는 로컬 LLM 폴백)


GROQ_MODEL = 'llama-3.1-8b-instant'
MAX_GEN_TOKEN = 512

if not USE_LOCAL_LLM:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)
else:
    # Groq API 키가 없을 때: 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
    # groq_client.chat.completions.create(...)와 동일한 인터페이스를 제공하는
    # 드롭인 클라이언트라, 이후 코드는 백엔드와 무관하게 그대로 동작한다.
    from types import SimpleNamespace

    from transformers import AutoModelForCausalLM, AutoTokenizer

    LOCAL_MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
    print(f'Groq API 키가 없어 로컬 LLM({LOCAL_MODEL_NAME})으로 생성합니다. '
          '최초 1회 모델 다운로드와 로딩에 시간이 걸릴 수 있습니다.')

    _local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    _local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME, torch_dtype=torch.float16,
    ).to(device)
    _local_model.eval()
    # VRAM이 빠듯하면(8GB 미만) [코드 12-14]의 BitsAndBytesConfig로 4비트 로드도 가능

    # Llama 3 계열 종료 토큰: 문장 끝 토큰과 <|eot_id|>를 함께 종료 신호로 사용
    _eot_id = _local_tokenizer.convert_tokens_to_ids('<|eot_id|>')
    _terminators = list({t for t in (_local_tokenizer.eos_token_id, _eot_id)
                         if isinstance(t, int) and t >= 0})

    class _LocalChat:
        @staticmethod
        def create(model, messages, max_tokens=MAX_GEN_TOKEN,
                   temperature=0.1, **kw):
            # 대화틀(chat template)로 messages를 모델 입력 형식으로 인코딩
            inputs = _local_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors='pt', return_dict=True,
            ).to(device)
            with torch.no_grad():
                output_ids = _local_model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,            # 일관된 결과를 위해 그리디 디코딩
                    eos_token_id=_terminators,
                    pad_token_id=_local_tokenizer.eos_token_id,
                )
            # 새로 생성된 토큰만 디코딩(입력 프롬프트 부분 제외)
            input_len = inputs['input_ids'].shape[-1]
            generated = output_ids[0][input_len:]
            text = _local_tokenizer.decode(generated, skip_special_tokens=True)
            return SimpleNamespace(
                choices=[SimpleNamespace(
                    message=SimpleNamespace(content=text.strip()))])

    class _LocalClient:
        def __init__(self):
            self.chat = SimpleNamespace(completions=_LocalChat())

    groq_client = _LocalClient()

Groq API 키가 없어 로컬 LLM(Bllossom/llama-3.2-Korean-Bllossom-3B)으로 생성합니다. 최초 1회 모델 다운로드와 로딩에 시간이 걸릴 수 있습니다.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<02:10,  1.94it/s]

Loading weights:  22%|██▏       | 57/254 [00:00<00:01, 120.69it/s]

...

Loading weights:  81%|████████▏ | 207/254 [00:01<00:00, 118.04it/s]

Loading weights:  87%|████████▋ | 220/254 [00:02<00:00, 106.13it/s]

Loading weights:  91%|█████████▏| 232/254 [00:02<00:00, 98.48it/s] 

Loading weights:  96%|█████████▌| 243/254 [00:02<00:00, 99.80it/s]

Loading weights: 100%|██████████| 254/254 [00:02<00:00, 108.51it/s]

- `chat.completions.create()`는 JSON 응답을 파이썬 객체로 감싸 반환한다.
    - 메시지 형식은 12-1절의 대화 메시지와 같다.

In [7]:
######################################################################################
# 코드 12-16 - Groq 클라이언트 생성과 호출
######################################################################################

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': '트랜스포머 모델은 어떤 구조인가요?'}],
    max_tokens=MAX_GEN_TOKEN,
    temperature=0.1,                 # 일관성을 위해 낮은 온도
)
print('생성 결과:\n' + response.choices[0].message.content)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


생성 결과:
트랜스포머 모델은 주로 자연어 처리(NLP) 분야에서 사용되는 deep learning 모델입니다. 이 모델은 주로 '트랜스포머(Transformer)'이라는 이름의 구조를 기반으로 합니다. 트랜스포머 모델은 주로 'Transformer' 논문에서 제안되었습니다. 이 모델은 주로 두 가지 주요 구조를 가지고 있습니다: Self-Attention Mechanism과 Multi-Head Attention Mechanism.

1. **Self-Attention Mechanism**:
   - **Self-Attention**: 이 메커니즘은 모델이 동일한 시퀀스 내에서 서로의 정보를 인식하고 연결하는 역할을 합니다. 예를 들어, 단어 간의 관계를 파악하는 데 사용됩니다. Self-Attention은 주로 두 개의 주어진 시퀀스 (예: 단어 시퀀스와 단어의 의미 시퀀스) 사이의 관계를 파악하는 데 사용됩니다.

2. **Multi-Head Attention Mechanism**:
   - **Multi-Head Attention**: Self-Attention 메커니즘을 여러 개의 주어진 시퀀스 (예: 단어 시퀀스와 단어의 의미 시퀀스) 사이의 관계를 파악하는 데 사용됩니다. Multi-Head Attention은 여러 개의 Self-Attention 메커니즘을 결합하여 더 강력한 정보를 추출합니다. 각 Self-Attention 메커니즘은 다른 시퀀스와의 관계를 파악하는 데 사용되며, 이를 통해 더 복잡한 관계를 파악할 수 있습니다.

트랜스포머 모델은 주로 두 가지 주요 구조를 가지고 있습니다:
- **Encoder**: 이 구조는 입력 시퀀스를 파악하는 역할을 합니다. Encoder는 Self-Attention 메커니즘을 사용하여 입력 시퀀스를 파악하고, 이를 통해 단어의 의미를 추출합니다.
- **Decoder**: 이 구조는 출력 시퀀스를 생성하는 역할을 합니다. Decoder는 Self-Attention 메커니즘을 사용하여 입력 시퀀스를 파악

- 시스템 메시지에는 어조와 안전 가이드라인을, 사용자 메시지에는 검색된 문서와 '그 문서를 참고해 답하라'는 지시를 담는다.
    - 근거 문서가 프롬프트에 함께 들어가므로, 모델은 학습하지 않은 정보로도 답할 수 있다.

In [8]:
######################################################################################
# 코드 12-17 - RAG 프롬프트 구성과 생성 함수
######################################################################################

def generate_with_groq(prompt, model=GROQ_MODEL, max_tokens=MAX_GEN_TOKEN):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system',
             'content': '전문성 있는 한국어 문장으로 답변하며, 자료가 제공되지 '
                        '않은 사항은 결코 추측해 답하지 말고 모른다고 응답한다.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return response.choices[0].message.content


def build_rag_prompt(query, retrieved_docs):
    # 검색된 문서를 컨텍스트로 결합
    context = '\n\n'.join([
        f"[문서 {i + 1}: {doc['title']}]\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    ])
    # 컨텍스트와 질문을 한 프롬프트로 묶고, 답변 지침을 맨 앞에 둔다.
    prompt = (
        '다음 문서에 없는 내용은 결코 추측해 답하지 말고 모른다고 답하며, '
        '문서를 참고해 질문에 답한다.\n\n'
        f'* 참고 문서\n{context}\n\n'
        f'* 질문\n{query}'
    )
    return prompt

- 검색 파이프라인과 생성 파이프라인을 한 함수로 묶는다.
    - `generate_fn`을 인자로 받아 생성 함수만 갈아 끼울 수 있게 한다.

In [9]:
######################################################################################
# 코드 12-18 - 통합 RAG 파이프라인
######################################################################################

def rag_pipeline(query, documents, doc_embeddings, embed_model,
                 generate_fn, top_k=2):
    # 1단계: 검색
    retrieved_docs, scores = retrieve(
        query, documents, doc_embeddings, embed_model, top_k=top_k,
    )
    # 2단계: 프롬프트 구성
    prompt = build_rag_prompt(query, retrieved_docs)
    # 3단계: 생성
    answer = generate_fn(prompt)
    return answer, retrieved_docs, scores

## RAG 적용 여부에 따른 답변 비교([표 12-11])

- 성격이 다른 세 질문으로 비교한다.
    1. 학습 시점 이후의 최신 정보를 묻는 질문
    2. 존재하지 않는 사실을 전제로 환각을 유도하는 질문
    3. 근거 문서의 범위를 벗어난 질문

In [10]:
# 참고 - RAG vs 비-RAG 답변 비교 ([표 12-11])
test_queries = [
    'Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?',       # 최신 정보
    'RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?',  # 환각 유도
    '파이토치와 텐서플로의 차이점은 무엇인가요?',                # 문서 범위 밖
]

for q in test_queries:
    print('=' * 60)
    print(f'질문: {q}\n')
    print('--- RAG 없이 (LLM 단독) ---')
    print(generate_with_groq(q))
    print('\n--- RAG 적용 ---')
    answer, docs, rag_scores = rag_pipeline(
        q, documents, doc_embeddings, embed_model, generate_with_groq,
    )
    titles = [d['title'] for d in docs]
    print(f'(검색된 문서: {titles}, '
          f'유사도: {[round(s, 3) for s in rag_scores]})')
    print(answer, '\n')

질문: Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?

--- RAG 없이 (LLM 단독) ---


Groq는 AI 기반의 데이터 모델링 플랫폼입니다. Groq는 데이터 모델링을 쉽고 빠르게 할 수 있도록 도와주는 기술을 제공합니다. Groq는 API를 통해 데이터 모델을 생성하고, 이를 통해 데이터베이스와 같은 데이터 스토어와 연동할 수 있습니다. Groq는 데이터 모델링을 자동화하는 데 도움을 주며, 이를 통해 데이터베이스 설계, 데이터 모델링, 데이터 전달 등 다양한 데이터 관리 작업을 효율적으로 수행할 수 있습니다.

--- RAG 적용 ---


(검색된 문서: ['Groq', 'GPT 시리즈'], 유사도: [0.477, 0.46])
Groq는 LPU(Language Processing Unit)라는 AI 추론 전용 칩을 개발한 스타트업입니다. Groq는 오픈소스 LLM의 API 서비스를 제공하며, 호출 인터페이스가 OpenAI API와 동일해 코드 호환성이 높습니다. 

질문: RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?

--- RAG 없이 (LLM 단독) ---


RAG(Reactive Application Gateway)는 Microsoft에서 제공하는 웹 애플리케이션 게이트웨이입니다. RAG는 2019년에 발표되었습니다. 

RAG는 파이프라인을 통해 여러 웹 애플리케이션을 관리하고, 트래픽을 분산하여 성능을 향상시키는 데 사용됩니다. RAG의 파이프라인 구성은 다음과 같습니다:

1. **트래픽 수집**: RAG는 HTTP 트래픽을 수집하여 각 애플리케이션에 전달합니다.
2. **트래픽 분산**: RAG는 트래픽을 여러 애플리케이션에 분산하여 성능을 향상시킵니다.
3. **응답 처리**: RAG는 각 애플리케이션의 응답을 수집하고, 응답을 전달합니다.
4. **데이터 분석**: RAG는 트래픽 데이터를 분석하여 성능을 모니터링하고, 최적화할 수 있는 정보를 제공합니다.

RAG는 Azure와 함께 사용될 수 있으며, Azure에서 구축할 수 있습니다. RAG는 여러 애플리케이션을 관리하는 데 유용하며, 트래픽을 효율적으로 분산하여 성능을 향상시킬 수 있습니다.

--- RAG 적용 ---


(검색된 문서: ['GPT 시리즈', 'RAG(검색 증강 생성)'], 유사도: [0.422, 0.345])
문서 2: RAG(검색 증강 생성)에서 RAG는 2020년 메타(Meta) AI 연구팀이 발표한 기법이라고 합니다. RAG의 파이프라인은 세 단계로 구성됩니다:

1. **문서 임베딩**: 문서를 임베딩 벡터로 변환해 저장합니다.
2. **질문 임베딩**: 질문을 임베딩해 유사한 문서를 검색합니다.
3. **LLM에 전달**: 검색된 문서를 컨텍스트로 LLM에 전달해 답변을 생성합니다.

따라서 RAG는 2020년 메타(Meta) AI 연구팀이 발표한 기법이며, 파이프라인은 세 단계로 구성되어 있습니다. 

질문: 파이토치와 텐서플로의 차이점은 무엇인가요?

--- RAG 없이 (LLM 단독) ---


파이토치(Python)와 텐서플로(TensorFlow)는 두 가지 주요 인공지능(IA) 프레임워크로, 각각의 특성과 사용 목적이 다릅니다. 다음은 두 프레임워크의 주요 차이점입니다:

1. **프로그래밍 언어**:
   - **파이토치**: 파이토치는 Python으로 프로그래밍합니다. Python은 간단하고 유연한 프로그래밍 언어로, 다양한 데이터 과학 및 IA 도구와 잘 통합됩니다.
   - **텐서플로**: 텐서플로는 C++로 프로그래밍합니다. C++는 성능과 효율성을 높이는 데 유리하지만, Python과 같은 간단한 프로그래밍 언어와는 다릅니다.

2. **사용 목적**:
   - **파이토치**: 파이토치는 다양한 IA 도구와 잘 통합되어 있어, 데이터 과학, 머신러닝, 딥러닝, 자연어 처리(NLP), 컴퓨터 비전 등 다양한 분야에서 사용됩니다. 또한, Python의 강력한 라이브러리와 모듈을 활용할 수 있어, 다양한 프로젝트에 적합합니다.
   - **텐서플로**: 텐서플로는 주로 딥러닝과 머신러닝에 중점을 둡니다. TensorFlow는 TensorFlow의 API를 통해 다양한 딥러닝 모델을 구현할 수 있게 해줍니다. 또한, TensorFlow의 성능과 효율성을 높이기 위해 C++로 프로그래밍합니다.

3. **성능**:
   - **파이토치**: 파이토치는 Python의 강력한 라이브러리와 모듈을 활용하여, 다양한 IA 도구와 잘 통합되어 있어 성능이 좋습니다. 그러나 C++로 프로그래밍하는 텐서플로와 비교하면 성능이 높을 수 있습니다.
   - **텐서플로**: 텐서플로는 C++로 프로그래밍하여, 매우 높은 성능을 제공합니다. 그러나 Python으로 프로그래밍하는 파이토치와 비교하면 성능이 낮을 수 있습니다.

4. **사용자 인터페이스**:
   - **파이토치**: 파이토치는 Python의 강력한 라이브러리와 모듈을 활용하여,

--- RAG 적용 ---


(검색된 문서: ['RAG(검색 증강 생성)', 'GPT 시리즈'], 유사도: [0.43, 0.413])
파이토치와 텐서플로의 차이점은 다음과 같습니다:

1. **개발자 및 사용자**: 파이토치는 주로 Python을 사용하는 개발자와 연구자들에 의해 개발되었습니다. 반면, 텐서플로는 Google의 TensorFlow 팀이 개발하고, 다양한 프로그래밍 언어를 지원합니다.

2. **사용 목적**: 파이토치는 주로 딥러닝 모델을 구축하고 학습하는 데 사용됩니다. 텐서플로는 TensorFlow와 같은 TensorFlow 2.x를 기반으로 한 딥러닝 프레임워크를 제공하며, 다양한 딥러닝 기술을 지원합니다.

3. **구성**: 파이토치는 TensorFlow와 유사한 구조를 가지고 있으며, TensorFlow의 API를 사용하여 모델을 구축할 수 있습니다. 텐서플로는 TensorFlow와의 호환성을 유지하면서도, TensorFlow 2.x와의 차이점을 보완한 API를 제공합니다.

4. **사용자 인터페이스**: 파이토치는 TensorFlow의 API를 사용하여 모델을 구축하고 학습할 수 있지만, 텐서플로는 TensorFlow 2.x와의 호환성을 유지하면서도, 더 간단하고 사용자 친화적인 인터페이스를 제공합니다.

5. **사용 사례**: 파이토치는 주로 Python을 사용하는 개발자와 연구자들에 의해 사용되며, 딥러닝 모델을 구축하고 학습하는 데 주로 사용됩니다. 텐서플로는 TensorFlow와의 호환성을 유지하면서도, 다양한 프로그래밍 언어를 지원하며, 다양한 딥러닝 기술을 지원합니다.

6. **라이브러리 및 프레임워크**: 파이토치는 TensorFlow와 유사한 라이브러리를 제공하며, TensorFlow의 API를 사용하여 모델을 구축할 수 있습니다. 텐서플로는 TensorFlow와의 호환성을 유지하면서도, TensorFlow 2.x와의 차이점을 보완한 라이브러리를 제공합니다.

7. **사용자 커뮤니티**: 파이토치는 TensorFlow와 유사한 커뮤니티를 

## 정리

- RAG는 질문과 관련된 문서를 검색해 프롬프트에 함께 넣는 방식으로, 학습 이후의 정보나 학습 데이터에 부족했던 도메인 지식을 채운다.
- 검색은 문장 임베딩의 코사인 유사도로 구현할 수 있다. 임베딩을 정규화해 두면 행렬곱 한 번으로 계산된다.
- 근거 문서를 함께 주면 환각이 줄어들지만, 문서 범위를 벗어난 질문에는 '모른다'고 답하도록 지시를 넣어야 한다.